# k09 — v1.2 extension: attention-based message passing (GAT)
Design of record: `STAGE2_DESIGN_ADDENDUM_v1.2_GAT.md` (frozen 2026-09-22, before this run). Specified **after** the original results were known; reported whatever the outcome.
- GAT and GAT-rewired: identity-free node features, same layer form as GraphSAGE with learned multi-head attention over observed transitions (log-multiplicity kept, so a=0 reduces to GraphSAGE).
- Same 8-configuration grid, two-fold recording-level cross-fitting on train_01 only, same selection and tie-break, 5 seeds, thresholds from out-of-fold scores.
- Test data read: the four B cells (test_02) only, SHA-256 verified; windows asserted identical to k05. Comparators: frozen k05 DeepSets and GraphSAGE scores.
- Inputs: k02 verified data cache (zip + hashes), k03 tuning outputs (fold check), k05 evaluation outputs (comparator scores).

In [ ]:
import os
os.makedirs('/kaggle/working/code', exist_ok=True); os.makedirs('/kaggle/temp', exist_ok=True)
FILES = {'feats.py': 'import numpy as np, pandas as pd, re, os, time\nW = 64\nHEXV = np.full(256, 0, dtype=np.uint8)\nfor i, ch in enumerate(\'0123456789abcdef\'):\n    HEXV[ord(ch)] = i; HEXV[ord(ch.upper())] = i\nPOP = np.array([bin(i).count(\'1\') for i in range(256)], dtype=np.uint8)\n\ndef slog(x):\n    return np.sign(x) * np.log1p(np.abs(x))\n\ndef load_file(path):\n    df = pd.read_csv(path, dtype={\'arbitration_id\': str, \'data_field\': str, \'attack\': np.int8}, keep_default_na=True)\n    ts = df[\'timestamp\'].to_numpy(np.float64)\n    ids = df[\'arbitration_id\'].str.rjust(3, \'0\').str[-3:]\n    ib = np.frombuffer(\'\'.join(ids.tolist()).encode(\'ascii\'), dtype=np.uint8).reshape(-1, 3)\n    idv = HEXV[ib].astype(np.int32)\n    id_int = idv[:, 0] * 256 + idv[:, 1] * 16 + idv[:, 2]\n    d = df[\'data_field\'].fillna(\'\')\n    plen = (d.str.len().to_numpy() // 2).clip(0, 8).astype(np.int8)\n    d16 = d.str[:16].str.ljust(16, \'0\')\n    db = np.frombuffer(\'\'.join(d16.tolist()).encode(\'ascii\'), dtype=np.uint8).reshape(-1, 16)\n    nib = HEXV[db]\n    pay = (nib[:, 0::2] * 16 + nib[:, 1::2]).astype(np.uint8)\n    posmask = np.arange(8)[None, :] < plen[:, None]\n    pay = np.where(posmask, pay, 0).astype(np.uint8)\n    y = df[\'attack\'].to_numpy(np.int8)\n    return ts, id_int, plen, pay, y\n\ndef per_frame_globals(ts, id_int, plen, pay):\n    n = len(ts); idx = np.arange(n)\n    order = np.lexsort((idx, id_int))\n    prev = np.full(n, -1, dtype=np.int64)\n    same = np.r_[False, id_int[order][1:] == id_int[order][:-1]]\n    prev[order[same]] = order[np.flatnonzero(same) - 1]\n    has = prev >= 0\n    pp = np.where(has, prev, 0)\n    dt_same = np.where(has, ts - ts[pp], 0.0)\n    x = pay ^ pay[pp]\n    ham = np.where(has, POP[x].sum(1), 0).astype(np.float32)\n    maxlen = np.maximum(plen, plen[pp]).astype(np.float32)\n    chg = np.where(has, (x != 0).sum(1) / np.maximum(maxlen, 1), 0).astype(np.float32)\n    lenchg = np.where(has, plen != plen[pp], False)\n    ent = np.zeros(n, dtype=np.float32)\n    for s in range(0, n, 500000):\n        b = pay[s:s + 500000]; L = plen[s:s + 500000].astype(np.int32)\n        valid = np.arange(8)[None, :] < L[:, None]\n        eq = (b[:, :, None] == b[:, None, :]) & valid[:, :, None] & valid[:, None, :]\n        c = eq.sum(2).astype(np.float32)\n        Lf = np.maximum(L, 1).astype(np.float32)[:, None]\n        with np.errstate(divide=\'ignore\', invalid=\'ignore\'):\n            term = np.where(valid, np.log2(np.where(c > 0, c, 1) / Lf), 0.0)\n        ent[s:s + 500000] = -(term.sum(1) / Lf[:, 0])\n    return prev, dt_same, ham, chg, ent, lenchg\n\nFRAME_NAMES = [\'plen\', \'dt_prev_any\', \'dt_same\', \'no_prev_same_in_window\', \'hamming\', \'changed_frac\', \'entropy\', \'same_as_prev\']\nNODE_NAMES = [\'count\', \'first_pos\', \'last_pos\', \'ia_mean\', \'ia_min\', \'ia_max\', \'ia_missing\', \'plen_mean\', \'plen_max\', \'plen_changes\', \'ham_mean\', \'chg_mean\', \'ent_mean\']\nGLOBAL_NAMES = [\'duration\', \'distinct_ids\', \'distinct_transitions\', \'fps\']\n\ndef windows_for_file(path, stride, id_perm=None):\n    ts, id_int, plen, pay, y = load_file(path)\n    if id_perm is not None:\n        id_int = id_perm[id_int]\n    n = len(ts)\n    if n < W:\n        return None\n    prev, dt_same_g, ham_g, chg_g, ent_g, lenchg_g = per_frame_globals(ts, id_int, plen, pay)\n    starts = np.arange(0, n - W + 1, stride)\n    nw = len(starts)\n    I = starts[:, None] + np.arange(W)[None, :]\n    ok = prev[I] >= starts[:, None]\n    tsw = ts[I]\n    dtp = np.diff(tsw, axis=1, prepend=tsw[:, :1])\n    idw = id_int[I]\n    fr = np.zeros((nw, W, len(FRAME_NAMES)), dtype=np.float32)\n    fr[..., 0] = plen[I] / 8.0\n    fr[..., 1] = slog(dtp * 1000)\n    fr[..., 2] = np.where(ok, slog(dt_same_g[I] * 1000), 0)\n    fr[..., 3] = ~ok\n    fr[..., 4] = np.where(ok, ham_g[I] / 64.0, 0)\n    fr[..., 5] = np.where(ok, chg_g[I], 0)\n    fr[..., 6] = ent_g[I] / 3.0\n    fr[:, 1:, 7] = idw[:, 1:] == idw[:, :-1]\n    # nodes\n    key = (np.arange(nw)[:, None] * 4096 + idw).ravel()\n    uk, inv = np.unique(key, return_inverse=True)\n    inv = inv.reshape(nw, W)\n    win_of_node = uk // 4096\n    node_first = np.searchsorted(win_of_node, np.arange(nw))\n    local = inv - node_first[:, None]\n    nn = np.bincount(win_of_node, minlength=nw)\n    G = len(uk)\n    fl = inv.ravel()\n    pos = np.broadcast_to(np.arange(W), (nw, W)).ravel()\n    okf = ok.ravel()\n    def agg_sum(v, m=None):\n        return np.bincount(fl, weights=(v if m is None else v * m), minlength=G)\n    order = np.argsort(fl, kind=\'stable\'); fs = fl[order]\n    bnd = np.flatnonzero(np.r_[True, fs[1:] != fs[:-1]])\n    def agg_min(v): return np.minimum.reduceat(v[order], bnd)\n    def agg_max(v): return np.maximum.reduceat(v[order], bnd)\n    cnt = np.bincount(fl, minlength=G).astype(np.float32)\n    iak = np.where(okf, slog(dt_same_g[I].ravel() * 1000), np.nan)\n    nia = agg_sum(okf.astype(np.float64))\n    has_ia = nia > 0\n    ia_mean = np.where(has_ia, agg_sum(np.nan_to_num(iak)) / np.maximum(nia, 1), 0)\n    ia_min = np.where(has_ia, agg_min(np.where(okf, iak, np.inf)), 0)\n    ia_max = np.where(has_ia, agg_max(np.where(okf, iak, -np.inf)), 0)\n    pl = (plen[I].ravel()).astype(np.float64)\n    nd = np.zeros((G, len(NODE_NAMES)), dtype=np.float32)\n    nd[:, 0] = cnt / W\n    nd[:, 1] = agg_min(pos.astype(np.float64)) / W\n    nd[:, 2] = agg_max(pos.astype(np.float64)) / W\n    nd[:, 3] = ia_mean; nd[:, 4] = ia_min; nd[:, 5] = ia_max\n    nd[:, 6] = ~has_ia\n    nd[:, 7] = agg_sum(pl) / cnt / 8.0\n    nd[:, 8] = agg_max(pl) / 8.0\n    nd[:, 9] = agg_max(pl) != agg_min(pl)\n    nd[:, 10] = np.where(has_ia, agg_sum(ham_g[I].ravel().astype(np.float64), okf) / np.maximum(nia, 1) / 64.0, 0)\n    nd[:, 11] = np.where(has_ia, agg_sum(chg_g[I].ravel().astype(np.float64), okf) / np.maximum(nia, 1), 0)\n    nd[:, 12] = agg_sum(ent_g[I].ravel().astype(np.float64)) / cnt / 3.0\n    node = np.zeros((nw, W, len(NODE_NAMES)), dtype=np.float32)\n    node[win_of_node, np.arange(G) - node_first[win_of_node]] = nd\n    # edges: local src/dst per transition\n    src = local[:, :-1].astype(np.uint8); dst = local[:, 1:].astype(np.uint8)\n    tr = np.unique((np.arange(nw)[:, None] * 4096 + local[:, :-1] * 64 + local[:, 1:]).ravel())\n    ntr = np.bincount(tr // 4096, minlength=nw)\n    dur = tsw[:, -1] - tsw[:, 0]\n    glob = np.stack([slog(dur * 1000), nn / W, ntr / 63.0, slog(W / np.maximum(dur, 1e-6))], 1).astype(np.float32)\n    lab = (y[I].max(1) > 0).astype(np.int8)\n    nattack = y[I].sum(1).astype(np.int16)\n    return dict(frame=fr.astype(np.float16), node=node.astype(np.float16), nmask=(np.arange(W)[None, :] < nn[:, None]),\n                src=src, dst=dst, glob=glob, y=lab, nattack=nattack, starts=starts.astype(np.int64), t0=tsw[:, 0], t1=tsw[:, -1])\n\nSTRUCT_NAMES = [\'transition_entropy\', \'unique_transition_ratio\', \'self_loop_ratio\', \'mean_out_degree\',\n                \'max_out_degree\', \'max_in_degree\', \'degree_entropy\', \'density\']\n\ndef structural_features(src, dst, nmask):\n    """Explicit structural/topological summaries of each window\'s directed transition multigraph.\n    src, dst: (nw, 63) local node indices of consecutive frames; nmask: (nw, 64) valid nodes.\n    Uses only ID-free graph structure (local node indices are arbitrary labels)."""\n    nw, E = src.shape\n    n = nmask.sum(1).astype(np.float64)\n    s = src.astype(np.int64); d = dst.astype(np.int64)\n    w = np.repeat(np.arange(nw), E)\n    key = w * 4096 + (s * 64 + d).ravel()\n    uk, cnt = np.unique(key, return_counts=True)\n    uw = uk // 4096; us = (uk % 4096) // 64; ud = (uk % 4096) % 64\n    p = cnt / float(E)\n    ent = np.bincount(uw, weights=-p * np.log2(p), minlength=nw) / np.log2(E)\n    uniq = np.bincount(uw, minlength=nw) / float(E)\n    selfr = (s == d).sum(1) / float(E)\n    ns = us != ud\n    outdeg = np.bincount(uw[ns] * 64 + us[ns], minlength=nw * 64).reshape(nw, 64).astype(np.float64)\n    indeg = np.bincount(uw[ns] * 64 + ud[ns], minlength=nw * 64).reshape(nw, 64).astype(np.float64)\n    e_ns = np.bincount(uw[ns], minlength=nw).astype(np.float64)\n    mean_out = np.where(n > 0, e_ns / np.maximum(n, 1), 0)\n    tot = outdeg + indeg; ts = tot.sum(1, keepdims=True)\n    pd_ = np.where(ts > 0, tot / np.maximum(ts, 1), 0)\n    with np.errstate(divide=\'ignore\', invalid=\'ignore\'):\n        h = -(np.where(pd_ > 0, pd_ * np.log2(np.where(pd_ > 0, pd_, 1)), 0)).sum(1)\n    deg_ent = np.where(n > 1, h / np.log2(np.maximum(n, 2)), 0)\n    dens = np.where(n > 1, e_ns / np.maximum(n * (n - 1), 1), 0)\n    return np.stack([ent, uniq, selfr, mean_out, outdeg.max(1), indeg.max(1), deg_ent, dens], 1).astype(np.float32)\n', 'models.py': 'import torch, torch.nn as nn, numpy as np, time\nW = 64\n\ndef mlp(i, h, o, drop):\n    return nn.Sequential(nn.Linear(i, h), nn.ReLU(), nn.Dropout(drop), nn.Linear(h, o))\n\nclass Head(nn.Module):\n    def __init__(self, h, g, drop):\n        super().__init__(); self.rho = mlp(3 * h + g, h, 1, drop)\n    def forward(self, H, mask, glob):\n        m = mask.unsqueeze(-1).float()\n        s = (H * m).sum(1); mean = s / m.sum(1).clamp(min=1)\n        mx = H.masked_fill(m == 0, -1e4).max(1).values\n        return self.rho(torch.cat([s, mean, mx, glob], 1)).squeeze(-1)\n\nclass DeepSets(nn.Module):\n    def __init__(self, f, h, g, drop=0.0):\n        super().__init__()\n        self.phi = nn.Sequential(nn.Linear(f, h), nn.ReLU(), nn.Dropout(drop), nn.Linear(h, h), nn.ReLU())\n        self.head = Head(h, g, drop)\n    def forward(self, b):\n        return self.head(self.phi(b[\'node\']), b[\'nmask\'], b[\'glob\'])\n\nclass SAGELayer(nn.Module):\n    def __init__(self, i, o):\n        super().__init__(); self.self_lin = nn.Linear(i, o); self.nei_lin = nn.Linear(i, o, bias=False)\n    def forward(self, H, A):\n        # A[b, src, dst] = weight; aggregate incoming neighbours of each dst node (weighted mean)\n        agg = torch.bmm(A.transpose(1, 2), H)\n        deg = A.sum(1).unsqueeze(-1)\n        agg = agg / deg.clamp(min=1e-9)\n        return self.self_lin(H) + self.nei_lin(agg)\n\nclass GraphSAGE(nn.Module):\n    def __init__(self, f, h, g, drop=0.0):\n        super().__init__()\n        self.l1 = SAGELayer(f, h); self.l2 = SAGELayer(h, h); self.drop = nn.Dropout(drop)\n        self.head = Head(h, g, drop)\n    def forward(self, b):\n        A = b[\'adj\']; m = b[\'nmask\'].unsqueeze(-1).float()\n        H = torch.relu(self.l1(b[\'node\'], A)) * m\n        H = torch.relu(self.l2(self.drop(H), A)) * m\n        return self.head(H, b[\'nmask\'], b[\'glob\'])\n\nclass GRUNet(nn.Module):\n    def __init__(self, f, h, g, drop=0.0):\n        super().__init__()\n        self.gru = nn.GRU(f, h, batch_first=True); self.out = mlp(2 * h + g, h, 1, drop)\n    def forward(self, b):\n        O, hn = self.gru(b[\'frame\'])\n        return self.out(torch.cat([hn[-1], O.mean(1), b[\'glob\']], 1)).squeeze(-1)\n\ndef nparams(m):\n    return sum(p.numel() for p in m.parameters())\n\ndef build_adj(src, dst, B, device):\n    A = torch.zeros(B, W, W, device=device)\n    bi = torch.arange(B, device=device).unsqueeze(1).expand_as(src)\n    A.index_put_((bi.reshape(-1), src.reshape(-1).long(), dst.reshape(-1).long()), torch.full((src.numel(),), 1.0 / 63, device=device), accumulate=True)\n    return A\n\ndef rewire_dst(dst, gen):\n    # degree-preserving directed rewiring: permute destination endpoints among a window\'s 63 edges\n    # (every source keeps its out-degree, every destination keeps its in-degree, multiplicities included)\n    r = torch.rand(dst.shape, generator=gen, device=dst.device)\n    perm = r.argsort(1)\n    return torch.gather(dst, 1, perm)\n\ndef edge_change_fraction(src, dst, dst2):\n    # fraction of the 63 directed edges (as a multiset per window) not present in the original\n    B = src.shape[0]\n    k1 = (src.long() * 64 + dst.long()).sort(1).values\n    k2 = (src.long() * 64 + dst2.long()).sort(1).values\n    fr = []\n    for i in range(B):\n        a, ca = torch.unique(k1[i], return_counts=True); b2, cb = torch.unique(k2[i], return_counts=True)\n        common = 0\n        d = dict(zip(a.tolist(), ca.tolist()))\n        for kk, cc in zip(b2.tolist(), cb.tolist()):\n            common += min(cc, d.get(kk, 0))\n        fr.append(1 - common / 63.0)\n    return float(np.mean(fr))\n\n\n# ---------------- v1.2 extension: attention-based message passing (STAGE2_DESIGN_ADDENDUM_v1.2_GAT) ----------------\nclass GATLayer(nn.Module):\n    """Same layer form as SAGELayer: out = Theta_s h_v + sum_u alpha_uv Theta_n h_u, with multi-head attention over the observed\n    incoming transitions u->v:  alpha_uv = softmax_u( LeakyReLU_0.2(a_src.Theta_n h_u + a_dst.Theta_n h_v) + log A_uv ).\n    With a = 0 this reduces exactly to SAGELayer\'s transition-weighted mean (log A_uv keeps the multiplicities).\n    Nodes with no incoming transition receive a zero neighbour message (as SAGELayer)."""\n    def __init__(self, i, o, heads=4):\n        super().__init__()\n        assert o % heads == 0\n        self.k = heads; self.d = o // heads\n        self.self_lin = nn.Linear(i, o); self.nei_lin = nn.Linear(i, o, bias=False)\n        self.a_src = nn.Parameter(torch.zeros(heads, self.d)); self.a_dst = nn.Parameter(torch.zeros(heads, self.d))\n        nn.init.xavier_uniform_(self.a_src); nn.init.xavier_uniform_(self.a_dst)\n    def forward(self, H, A):\n        B, N, _ = H.shape\n        Wh = self.nei_lin(H).view(B, N, self.k, self.d)                      # (B,N,K,d)\n        es = (Wh * self.a_src).sum(-1); ed = (Wh * self.a_dst).sum(-1)       # (B,N,K)\n        logits = torch.nn.functional.leaky_relu(es.unsqueeze(2) + ed.unsqueeze(1), 0.2)   # (B,u,v,K)\n        edge = (A > 0).unsqueeze(-1)\n        logits = logits + torch.log(A.clamp(min=1e-12)).unsqueeze(-1)\n        logits = logits.masked_fill(~edge, -1e9)\n        alpha = torch.softmax(logits, dim=1) * edge.float()                  # softmax over sources u; zero where no edge\n        msg = torch.einsum(\'buvk,bukd->bvkd\', alpha, Wh).reshape(B, N, self.k * self.d)\n        return self.self_lin(H) + msg\n\nclass GAT(nn.Module):\n    def __init__(self, f, h, g, drop=0.0, heads=4):\n        super().__init__()\n        self.l1 = GATLayer(f, h, heads); self.l2 = GATLayer(h, h, heads); self.drop = nn.Dropout(drop)\n        self.head = Head(h, g, drop)\n    def forward(self, b):\n        A = b[\'adj\']; m = b[\'nmask\'].unsqueeze(-1).float()\n        H = torch.relu(self.l1(b[\'node\'], A)) * m\n        H = torch.relu(self.l2(self.drop(H), A)) * m\n        return self.head(H, b[\'nmask\'], b[\'glob\'])\n', 'exp_gat.py': '"""k09: v1.2 extension (STAGE2_DESIGN_ADDENDUM_v1.2_GAT). GAT and GAT-rewired: tuning (train_01 only), final models (5 seeds),\nthresholds from out-of-fold scores, and scoring of the B cells (test_02) only. Identical rules to k03/k04/k05 v2.\nUsage: python exp_gat.py --sets set_01,set_03 --device cuda:0 [--smoke]"""\nimport os, sys, re, json, time, glob, hashlib, argparse, traceback, zipfile, math\nimport numpy as np, torch\nfrom sklearn.metrics import average_precision_score, log_loss, precision_recall_curve\nsys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))\nimport feats, models\n\nap = argparse.ArgumentParser()\nap.add_argument(\'--sets\', required=True); ap.add_argument(\'--device\', default=\'cuda:0\')\nap.add_argument(\'--out\', default=\'/kaggle/working/gat\'); ap.add_argument(\'--seeds\', default=\'0,1,2,3,4\')\nap.add_argument(\'--smoke\', action=\'store_true\', help=\'tiny run (2 configs, 1 epoch, 1 seed, 2 test files) to check the pipeline; writes to <out>_smoke\')\nargs = ap.parse_args()\nDEV = args.device; SETS = args.sets.split(\',\'); SEEDS = [int(s) for s in args.seeds.split(\',\')]\nOUT = args.out + (\'_smoke\' if args.smoke else \'\'); os.makedirs(OUT, exist_ok=True)\nEPOCH_CAP, PATIENCE, BS = (1, 1, 1024) if args.smoke else (20, 3, 1024)\nif args.smoke: SEEDS = SEEDS[:1]\nZIP = glob.glob(\'/kaggle/input/**/can-train-and-test-v1.zip\', recursive=True)[0]\nHASHES = glob.glob(\'/kaggle/input/**/file_hashes_sha256.csv\', recursive=True)[0]\nDATA = f\'/kaggle/temp/gat_{"_".join(SETS)}\'\nEVAL_REWIRE_SEED = 12345; CONST_STD = 1.5e-6\nMODELS = (\'GAT\', \'GAT_rewired\')\nLOG = open(os.path.join(OUT, f\'log_{"_".join(SETS)}.txt\'), \'a\')\ndef log(*a):\n    s = time.strftime(\'%H:%M:%S \') + \' \'.join(str(x) for x in a); print(s, flush=True); LOG.write(s + \'\\n\'); LOG.flush()\n\nH = {}\nfor line in open(HASHES).read().strip().split(\'\\n\')[1:]:\n    rel, size, sha = line.split(\',\'); H[rel] = (int(size), sha)\ndef fam(n): return re.sub(r\'-\\d+\\.csv$\', \'\', os.path.basename(n))\ndef idx(p): return int(re.search(r\'-(\\d+)\\.csv$\', p).group(1))\ndef sha_of(p):\n    hh = hashlib.sha256()\n    with open(p, \'rb\') as f:\n        for b in iter(lambda: f.read(8 << 20), b\'\'): hh.update(b)\n    return hh.hexdigest()\n\nNF, NN, NG = len(feats.FRAME_NAMES), len(feats.NODE_NAMES), len(feats.GLOBAL_NAMES)\ndef make(name, h, drop):\n    if name == \'GraphSAGE\': return models.GraphSAGE(NN, h, NG, drop)\n    return models.GAT(NN, h, NG, drop)\n_MH = {}\ndef matched_hidden(h_sage):\n    """GAT width (divisible by 4 heads) whose trainable-parameter count is closest to GraphSAGE\'s at that grid point; must be within 10%."""\n    if h_sage in _MH: return _MH[h_sage]\n    target = models.nparams(make(\'GraphSAGE\', h_sage, 0.0))\n    h = min(range(4, 513, 4), key=lambda x: abs(models.nparams(make(\'GAT\', x, 0.0)) - target))\n    assert abs(models.nparams(make(\'GAT\', h, 0.0)) - target) <= 0.10 * target, \'parameter matching outside 10%\'\n    _MH[h_sage] = h; return h\nNEURAL_GRID = [dict(lr=lr, h_sage=h, dropout=dr) for lr in (1e-3, 3e-4) for h in (32, 64) for dr in (0.0, 0.2)]\nif args.smoke: NEURAL_GRID = NEURAL_GRID[:2]\n\ndef extract(z, names):\n    out = []\n    for n in names:\n        p = os.path.join(DATA, n)\n        if not os.path.exists(p): z.extract(n, DATA)\n        rel = n.split(\'can-train-and-test/\')[1]\n        assert (os.path.getsize(p), sha_of(p)) == H[rel], \'hash mismatch \' + rel\n        out.append(p)\n    return out\n\nKEYS = [\'node\', \'nmask\', \'src\', \'dst\', \'glob\', \'y\']\ndef cat(F, plist, stride64):\n    out = {k: [] for k in KEYS}\n    for p in plist:\n        d = F[p]; sel = (d[\'starts\'] % 64 == 0) if stride64 else slice(None)\n        for k in out: out[k].append(d[k][sel])\n    return {k: np.concatenate(v) for k, v in out.items()}\n\ndef norm_stats(D, Fr=None):\n    msk = D[\'nmask\'].reshape(-1)\n    n = D[\'node\'].astype(np.float32).reshape(-1, NN)[msk]\n    return {\'fm\': np.zeros(NF, np.float32), \'fs\': np.ones(NF, np.float32), \'nm\': n.mean(0), \'ns\': n.std(0) + 1e-6, \'gm\': D[\'glob\'].mean(0), \'gs\': D[\'glob\'].std(0) + 1e-6}\n\ndef to_gpu(D, N, zero_const):\n    """float32 throughout; at test (zero_const=True) features constant in training are set to 0, exactly as k05 v2."""\n    T = lambda a: torch.tensor(a, dtype=torch.float32, device=DEV)\n    fm, fs, nm_, ns, gm, gs = T(N[\'fm\']), T(N[\'fs\']), T(N[\'nm\']), T(N[\'ns\']), T(N[\'gm\']), T(N[\'gs\'])\n    out = {}; msk = torch.from_numpy(D[\'nmask\']).to(DEV)\n    out[\'node\'] = ((torch.from_numpy(D[\'node\'].astype(np.float32)).to(DEV) - nm_) / ns) * msk.unsqueeze(-1)\n    out[\'glob\'] = (torch.from_numpy(D[\'glob\'].astype(np.float32)).to(DEV) - gm) / gs\n    if zero_const:\n        out[\'node\'] = out[\'node\'] * (ns > CONST_STD).float(); out[\'glob\'] = out[\'glob\'] * (gs > CONST_STD).float()\n    out[\'nmask\'] = msk; out[\'src\'] = torch.from_numpy(D[\'src\']).to(DEV); out[\'dst\'] = torch.from_numpy(D[\'dst\']).to(DEV)\n    out[\'y\'] = torch.from_numpy(D[\'y\'].astype(np.float32)).to(DEV); out[\'n\'] = len(D[\'y\'])\n    return out\n\ndef batch(T, ix, rewire, gen):\n    b = {\'node\': T[\'node\'][ix], \'nmask\': T[\'nmask\'][ix], \'glob\': T[\'glob\'][ix]}\n    dst = T[\'dst\'][ix]\n    if rewire: dst = models.rewire_dst(dst, gen)\n    b[\'adj\'] = models.build_adj(T[\'src\'][ix], dst, len(ix), DEV)\n    return b\n\ndef score(m, T, rewire):\n    m.eval(); g = torch.Generator(device=DEV); g.manual_seed(EVAL_REWIRE_SEED); out = []\n    with torch.no_grad():\n        for s in range(0, T[\'n\'], 2048):\n            ix = torch.arange(s, min(s + 2048, T[\'n\']), device=DEV)\n            out.append(m(batch(T, ix, rewire, g)).float())\n    return torch.cat(out).cpu().numpy().astype(np.float32)\n\ndef fit(name, cfg, h, Ttr, epochs, seed, Tva=None, yva=None):\n    """Train; with Tva returns early-stopping history (tuning), else trains exactly `epochs` epochs (final)."""\n    torch.manual_seed(seed); np.random.seed(seed)\n    gen = torch.Generator(device=DEV); gen.manual_seed(seed)\n    m = make(\'GAT\', h, cfg[\'dropout\']).to(DEV)\n    opt = torch.optim.AdamW(m.parameters(), lr=cfg[\'lr\'])\n    pos = float(Ttr[\'y\'].sum()); neg = Ttr[\'n\'] - pos\n    lossf = torch.nn.BCEWithLogitsLoss(pos_weight=torch.tensor(neg / max(pos, 1.0), device=DEV))\n    rew = name == \'GAT_rewired\'; hist = []; best = (-1, None, None, None); bad = 0; losses = []\n    for ep in range(epochs):\n        m.train(); t = time.time(); perm = torch.randperm(Ttr[\'n\'], device=DEV, generator=gen); tot = 0.0\n        for s in range(0, Ttr[\'n\'], BS):\n            ix = perm[s:s + BS]\n            loss = lossf(m(batch(Ttr, ix, rew, gen)), Ttr[\'y\'][ix]); opt.zero_grad(); loss.backward(); opt.step(); tot += loss.item() * len(ix)\n        losses.append(round(tot / Ttr[\'n\'], 6))\n        if Tva is None: continue\n        sc = score(m, Tva, rew)\n        apv = float(average_precision_score(yva, sc)) if yva.sum() > 0 else float(\'nan\')\n        bce = float(log_loss(yva, 1 / (1 + np.exp(-np.clip(sc, -30, 30))), labels=[0, 1]))\n        hist.append({\'epoch\': ep + 1, \'val_ap\': round(apv, 6), \'val_bce\': round(bce, 6), \'s\': round(time.time() - t, 1)})\n        if apv > best[0]: best = (apv, ep + 1, bce, sc); bad = 0\n        else:\n            bad += 1\n            if bad >= PATIENCE: break\n    if Tva is None: return m, losses\n    return {\'hidden\': h, \'params\': models.nparams(m), \'best_ap\': best[0], \'best_epoch\': best[1], \'best_bce\': best[2], \'history\': hist}, best[3]\n\ndef select(runs):\n    rows = []\n    for ci, cfg in enumerate(NEURAL_GRID):\n        r = [runs[ci][f] for f in (1, 2)]\n        if any(x is None for x in r): continue\n        rows.append(dict(ci=ci, ap=np.mean([x[\'best_ap\'] for x in r]), bce=np.mean([x[\'best_bce\'] for x in r]),\n                         ep=np.mean([x[\'best_epoch\'] for x in r]), size=cfg[\'h_sage\'], lr=cfg[\'lr\']))\n    best_ap = max(r[\'ap\'] for r in rows); tied = [r for r in rows if r[\'ap\'] >= best_ap - 0.001]\n    tied.sort(key=lambda r: (r[\'bce\'], r[\'ep\'], r[\'size\'], r[\'lr\'])); ch = tied[0]\n    return {\'config_index\': ch[\'ci\'], \'config\': NEURAL_GRID[ch[\'ci\']], \'mean_val_ap\': ch[\'ap\'], \'mean_val_bce\': ch[\'bce\'],\n            \'final_epochs\': max(1, int(round(ch[\'ep\']))), \'n_tied_within_0.001\': len(tied), \'all\': rows}\n\ndef thresholds_from_oof(scores, y, hours):   # identical to k04\n    neg = np.sort(scores[y == 0])[::-1]; pos = scores[y == 1]\n    out = {\'val_hours\': hours, \'n_val_windows\': int(len(y)), \'n_val_pos\': int(y.sum())}\n    p, r, t = precision_recall_curve(y, scores); f1 = 2 * p[:-1] * r[:-1] / np.maximum(p[:-1] + r[:-1], 1e-12); i = int(np.argmax(f1))\n    out[\'f1\'] = {\'threshold\': float(t[i]), \'op\': \'>=\', \'oof_f1\': float(f1[i])}\n    for rate in (1, 5):\n        k = int(math.floor(rate * hours)); thr = float(neg[k]) if k < len(neg) else float(neg[-1]) - 1e-6\n        out[f\'fa{rate}\'] = {\'threshold\': thr, \'op\': \'>\', \'allowed_fp\': k, \'oof_fp\': int((neg > thr).sum()), \'oof_recall\': float((pos > thr).mean()) if len(pos) else float(\'nan\')}\n    return out\n\ndef run_set(st):\n    sd = os.path.join(OUT, st); os.makedirs(sd, exist_ok=True)\n    if os.path.exists(os.path.join(sd, \'DONE\')): log(st, \'already done\'); return\n    train_sha = {H[r][1] for r in H if r.startswith(f\'{st}/train_01/\')}\n    with zipfile.ZipFile(ZIP) as z:\n        allnames = z.namelist()\n        tr_names = sorted(n for n in allnames if n.startswith(f\'can-train-and-test/{st}/train_01/\') and n.endswith(\'.csv\'))\n        t0 = time.time(); files = extract(z, tr_names); log(st, \'train files\', len(files), \'extract+verify s\', round(time.time() - t0, 1))\n        # ---- folds: identical rule to k03; checked against k03 meta when available\n        lo = min(idx(p) for p in files)\n        folds = {1: ([p for p in files if idx(p) == lo], [p for p in files if idx(p) != lo]),\n                 2: ([p for p in files if idx(p) != lo], [p for p in files if idx(p) == lo])}\n        k03 = glob.glob(f\'/kaggle/input/**/tune/{st}/meta.json\', recursive=True)\n        if k03:\n            M = json.load(open(k03[0]))\n            for f in (1, 2):\n                assert sorted(M[\'folds\'][str(f)][\'train\']) == sorted(os.path.basename(p) for p in folds[f][0]), \'fold mismatch vs k03\'\n            log(st, \'folds identical to k03\')\n        t0 = time.time(); F = {p: feats.windows_for_file(p, 32) for p in files}\n        hours = sum(float(F[p][\'t1\'].max() - F[p][\'t0\'].min()) / 3600.0 for p in files)\n        log(st, \'features s\', round(time.time() - t0, 1), \'train hours\', round(hours, 3))\n        # ---- tuning (train_01 only)\n        FD = {}\n        for f in (1, 2):\n            Dtr, Dva = cat(F, folds[f][0], False), cat(F, folds[f][1], True); N = norm_stats(Dtr)\n            FD[f] = (to_gpu(Dtr, N, False), to_gpu(Dva, N, False), Dva[\'y\'].astype(np.int8))\n        runs = {ci: {} for ci in range(len(NEURAL_GRID))}; scs = {ci: {} for ci in range(len(NEURAL_GRID))}\n        for ci, cfg in enumerate(NEURAL_GRID):\n            h = matched_hidden(cfg[\'h_sage\'])\n            for f in (1, 2):\n                t = time.time()\n                try:\n                    r, sc = fit(\'GAT\', cfg, h, FD[f][0], EPOCH_CAP, 0, FD[f][1], FD[f][2]); runs[ci][f] = r; scs[ci][f] = sc\n                except Exception as e:\n                    runs[ci][f] = None; log(st, \'GAT\', ci, f, \'ERROR\', repr(e), traceback.format_exc()[-800:])\n                log(st, \'GAT cfg\', ci, \'fold\', f, runs[ci][f] and {k: runs[ci][f][k] for k in (\'best_ap\', \'best_epoch\', \'params\')}, round(time.time() - t, 1), \'s\')\n        SEL = {\'GAT\': select(runs)}; SEL[\'GAT\'][\'hidden\'] = matched_hidden(SEL[\'GAT\'][\'config\'][\'h_sage\'])\n        OOF = {\'GAT\': scs[SEL[\'GAT\'][\'config_index\']], \'GAT_rewired\': {}}\n        cfg = SEL[\'GAT\'][\'config\']; h = SEL[\'GAT\'][\'hidden\']; rr = {}\n        for f in (1, 2):\n            r, sc = fit(\'GAT_rewired\', cfg, h, FD[f][0], EPOCH_CAP, 0, FD[f][1], FD[f][2]); rr[f] = r; OOF[\'GAT_rewired\'][f] = sc\n            log(st, \'GAT_rewired fold\', f, r[\'best_ap\'])\n        SEL[\'GAT_rewired\'] = {\'config\': cfg, \'hidden\': h, \'final_epochs\': SEL[\'GAT\'][\'final_epochs\'], \'fold_best_ap\': {f: rr[f][\'best_ap\'] for f in (1, 2)},\n                              \'note\': \'uses GAT selection (addendum v1.2 §3)\'}\n        json.dump({\'runs\': runs, \'runs_rewired\': rr}, open(os.path.join(sd, \'tune_runs.json\'), \'w\'), default=str)\n        json.dump(SEL, open(os.path.join(sd, \'selection.json\'), \'w\'), indent=1, default=str)\n        log(st, \'GAT selected\', SEL[\'GAT\'][\'config\'], \'hidden\', h, \'val AP\', round(SEL[\'GAT\'][\'mean_val_ap\'], 4), \'epochs\', SEL[\'GAT\'][\'final_epochs\'], \'ties\', SEL[\'GAT\'][\'n_tied_within_0.001\'])\n        y_oof = np.concatenate([FD[1][2], FD[2][2]]).astype(np.int8)\n        TH = {nm: thresholds_from_oof(np.concatenate([OOF[nm][1], OOF[nm][2]]).astype(np.float64), y_oof, hours) for nm in MODELS}\n        json.dump(TH, open(os.path.join(sd, \'thresholds.json\'), \'w\'), indent=1)\n        del FD; torch.cuda.empty_cache()\n        # ---- final models (whole train_01)\n        Dall = cat(F, files, False); N = norm_stats(Dall); np.savez(os.path.join(sd, \'norm_stats.npz\'), **N)\n        Tall = to_gpu(Dall, N, False); FIN = {}; META = {\'set\': st, \'train_windows\': int(len(Dall[\'y\'])), \'train_hours\': hours, \'models\': {}}\n        for nm in MODELS:\n            FIN[nm] = []; rec = {\'config\': cfg, \'hidden\': h, \'epochs\': SEL[\'GAT\'][\'final_epochs\'], \'seeds\': {}}\n            for seed in SEEDS:\n                t = time.time(); m, losses = fit(nm, cfg, h, Tall, SEL[\'GAT\'][\'final_epochs\'], seed)\n                torch.save(m.state_dict(), os.path.join(sd, f\'{nm}_seed{seed}.pt\')); m.eval(); FIN[nm].append(m)\n                rec[\'seeds\'][seed] = {\'train_loss\': losses, \'fit_s\': round(time.time() - t, 1), \'params\': models.nparams(m)}\n                log(st, nm, \'seed\', seed, \'params\', models.nparams(m), \'last loss\', losses[-1], \'fit s\', rec[\'seeds\'][seed][\'fit_s\'])\n            META[\'models\'][nm] = rec\n        json.dump(META, open(os.path.join(sd, \'final_meta.json\'), \'w\'), indent=1, default=str)\n        del Tall, Dall, F; torch.cuda.empty_cache()\n        # ---- B cell (test_02) scoring: the only test data read by this script\n        te_names = sorted(n for n in allnames if n.startswith(f\'can-train-and-test/{st}/test_02_\') and n.endswith(\'.csv\'))\n        if args.smoke: te_names = te_names[:2]\n        parts = {k: [] for k in [\'node\', \'nmask\', \'src\', \'dst\', \'glob\', \'y\', \'starts\']}; fid = []; finfo = []\n        for i, n in enumerate(te_names):\n            p = extract(z, [n])[0]; rel = n.split(\'can-train-and-test/\')[1]; sha = H[rel][1]\n            assert sha not in train_sha, \'LEAKAGE: test file identical to a train_01 file \' + rel\n            d = feats.windows_for_file(p, 64); os.remove(p); assert (d[\'starts\'] % 64 == 0).all()\n            for k in parts: parts[k].append(d[k])\n            fid.append(np.full(len(d[\'y\']), i, np.int16))\n            finfo.append({\'file\': os.path.basename(n), \'relative_path\': rel, \'sha256\': sha, \'token\': fam(n), \'windows\': int(len(d[\'y\'])),\n                          \'pos\': int(d[\'y\'].sum()), \'hours\': float(d[\'t1\'].max() - d[\'t0\'].min()) / 3600.0})\n        D = {k: np.concatenate(v) for k, v in parts.items()}; fid = np.concatenate(fid)\n        T = to_gpu(D, N, True)\n        S = {nm: np.stack([score(m, T, nm == \'GAT_rewired\') for m in FIN[nm]]) for nm in MODELS}\n        for nm in MODELS: assert np.isfinite(S[nm]).all(), f\'non-finite scores {st} {nm}\'\n        # windows must be identical to k05\'s B-cell windows (same comparators)\n        k05 = glob.glob(f\'/kaggle/input/**/eval/{st}/test_02_scores.npz\', recursive=True)\n        if k05 and not args.smoke:\n            Z = np.load(k05[0])\n            assert np.array_equal(Z[\'y\'], D[\'y\'].astype(np.int8)) and np.array_equal(Z[\'file_id\'], fid) and np.array_equal(Z[\'starts\'], D[\'starts\']), \'window mismatch vs k05\'\n            log(st, \'B-cell windows identical to k05\')\n        np.savez_compressed(os.path.join(sd, \'test_02_scores.npz\'), y=D[\'y\'].astype(np.int8), file_id=fid, starts=D[\'starts\'], **{f\'score_{nm}\': S[nm] for nm in MODELS})\n        json.dump({\'set\': st, \'cell\': \'test_02\', \'files\': finfo, \'windows\': int(len(D[\'y\'])), \'pos\': int(D[\'y\'].sum()), \'prevalence\': float(D[\'y\'].mean()),\n                   \'hours\': float(sum(f[\'hours\'] for f in finfo))}, open(os.path.join(sd, \'test_02_meta.json\'), \'w\'), indent=1)\n        if args.smoke: log(st, \'SMOKE: test_02 pipeline ran; score shapes\', {nm: S[nm].shape for nm in MODELS}, \'(no metric computed)\')\n        else: log(st, \'test_02 scored\', {nm: round(float(np.mean([average_precision_score(D[\'y\'], s) for s in S[nm]])), 4) for nm in MODELS})\n    open(os.path.join(sd, \'DONE\'), \'w\').write(\'ok\')\n    log(st, \'DONE\')\n\nfor st in SETS:\n    try:\n        run_set(st)\n    except Exception as e:\n        log(st, \'FATAL\', repr(e), traceback.format_exc()[-2000:])\n', 'gat_stats.py': '"""k09 statistics (STAGE2_DESIGN_ADDENDUM_v1.2_GAT §5). Reads k09 GAT/GAT_rewired B-cell scores and the FROZEN k05 B-cell scores of\nDeepSets and GraphSAGE (identical windows, asserted). Exact enumeration of the attack-family-stratified recording bootstrap (k06 code),\nB summary from 200,000 draws of the exact per-cell distributions, Holm over the three B-summary tests. No fitting, no selection.\nUsage: python gat_stats.py --gat /kaggle/working/gat --k05_eval <k05 eval dir> --out /kaggle/working/gat_stats"""\nimport os, sys, json, glob, argparse, math\nimport numpy as np\nfrom sklearn.metrics import average_precision_score\nsys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))\nfrom exp_audit import ap_prep, ap_w, exact_token_resamples, summarise\n\nMODELS = [\'DeepSets\', \'GraphSAGE\', \'GAT\', \'GAT_rewired\']\nCONTRASTS = {\'E1_GAT_minus_DeepSets\': (\'GAT\', \'DeepSets\'), \'E1a_GAT_minus_rewired\': (\'GAT\', \'GAT_rewired\'), \'E2_GAT_minus_GraphSAGE\': (\'GAT\', \'GraphSAGE\')}\nMC_SUMMARY = 200000; MC_SEED = 20260917\n\ndef apply_thr(s, thr): return (s >= thr[\'threshold\']) if thr[\'op\'] == \'>=\' else (s > thr[\'threshold\'])\ndef macro_f1(y, pred):\n    out = []\n    for c in (1, 0):\n        tp = np.sum((pred == c) & (y == c)); fp = np.sum((pred == c) & (y != c)); fn = np.sum((pred != c) & (y == c))\n        out.append(2 * tp / max(2 * tp + fp + fn, 1))\n    return float(np.mean(out))\ndef holm(p):\n    ks = sorted(p, key=lambda k: p[k]); m = len(ks); adj = {}; run = 0.0\n    for i, k in enumerate(ks): run = max(run, min(1.0, (m - i) * p[k])); adj[k] = run\n    return adj\n\ndef main(GAT, K05, OUT):\n    os.makedirs(OUT, exist_ok=True); R = {\'cells\': {}, \'summary\': {}, \'holm_B_summary\': {}}; dist = {}\n    for path in sorted(glob.glob(os.path.join(GAT, \'set_*\', \'test_02_scores.npz\'))):\n        st = path.split(os.sep)[-2]; key = f\'{st}/test_02\'\n        G = np.load(path); meta = json.load(open(path.replace(\'_scores.npz\', \'_meta.json\')))\n        K = np.load(os.path.join(K05, st, \'test_02_scores.npz\'))\n        y = G[\'y\'].astype(np.int8); fid = G[\'file_id\'].astype(np.int64)\n        assert np.array_equal(K[\'y\'], G[\'y\']) and np.array_equal(K[\'file_id\'], G[\'file_id\']) and np.array_equal(K[\'starts\'], G[\'starts\']), \'window mismatch\'\n        S = {\'DeepSets\': K[\'score_DeepSets\'], \'GraphSAGE\': K[\'score_GraphSAGE\'], \'GAT\': G[\'score_GAT\'], \'GAT_rewired\': G[\'score_GAT_rewired\']}\n        S = {m: v.astype(np.float64) for m, v in S.items()}\n        files = meta[\'files\']; nf = len(files); tokens = [f[\'token\'] for f in files]; hours = meta[\'hours\']\n        TH = json.load(open(os.path.join(GAT, st, \'thresholds.json\')))\n        c = {\'windows\': int(len(y)), \'pos\': int(y.sum()), \'prevalence\': float(y.mean()), \'hours\': hours, \'models\': {}, \'exact_bootstrap\': {}, \'leave_one_out\': {}}\n        for m in MODELS:\n            aps = [float(average_precision_score(y, s)) for s in S[m]]\n            mr = {\'ap_mean\': float(np.mean(aps)), \'ap_sd\': float(np.std(aps, ddof=1)) if len(aps) > 1 else 0.0, \'ap_seeds\': aps}\n            if m in TH:   # operating points only for the new models (k05 already reports the others)\n                th = TH[m]; mf = []; rec = {1: [], 5: []}; fah = {1: [], 5: []}\n                for s in S[m]:\n                    mf.append(macro_f1(y, apply_thr(s, th[\'f1\']).astype(np.int8)))\n                    for r in (1, 5):\n                        pr = apply_thr(s, th[f\'fa{r}\']); rec[r].append(float(np.mean(pr[y == 1]))); fah[r].append(int(np.sum(pr & (y == 0))) / hours)\n                mr.update({\'macro_f1_mean\': float(np.mean(mf)), **{f\'recall_at_{r}FAh_mean\': float(np.mean(rec[r])) for r in (1, 5)},\n                           **{f\'achieved_FAh_at_{r}FAh_mean\': float(np.mean(fah[r])) for r in (1, 5)}})\n            fam = {}\n            for t in sorted(set(tokens)):\n                ti = np.isin(fid, [i for i, tt in enumerate(tokens) if tt == t]); npos = int(y[ti].sum())\n                if npos >= 50: fam[t] = {\'pos\': npos, \'ap_token_recordings_mean\': float(np.mean([average_precision_score(y[ti], s[ti]) for s in S[m]]))}\n            mr[\'per_token\'] = fam; c[\'models\'][m] = mr\n        agg = {m: c[\'models\'][m][\'ap_mean\'] for m in MODELS}\n        c[\'point\'] = {h: agg[a] - agg[b] for h, (a, b) in CONTRASTS.items()}\n        for i, f in enumerate(files):\n            keep = fid != i\n            if y[keep].sum() == 0: continue\n            a2 = {m: float(np.mean([average_precision_score(y[keep], s[keep]) for s in S[m]])) for m in MODELS}\n            c[\'leave_one_out\'][f[\'file\']] = {h: round(a2[a] - a2[b], 4) for h, (a, b) in CONTRASTS.items()}\n        W, P = exact_token_resamples(tokens); B = {}\n        for m in MODELS:\n            acc = np.zeros(len(W))\n            for s in S[m]:\n                CP, CN = ap_prep(s, y, fid, nf)\n                assert abs(ap_w(CP, CN, np.ones((1, nf)))[0] - average_precision_score(y, s)) < 1e-9\n                acc += ap_w(CP, CN, W)\n            B[m] = acc / len(S[m])\n        dist[key] = {\'P\': P, \'ap\': B}\n        for h, (a, b) in CONTRASTS.items():\n            c[\'exact_bootstrap\'][h] = {\'point_full_sample\': round(c[\'point\'][h], 4), **summarise(B[a] - B[b], P)}\n        R[\'cells\'][key] = c\n        print(key, {m: round(agg[m], 4) for m in MODELS}, {h: v[\'decision\'] for h, v in c[\'exact_bootstrap\'].items()}, flush=True)\n    keys = sorted(dist); rng = np.random.default_rng(MC_SEED)\n    draws = {k: rng.choice(len(dist[k][\'P\']), size=MC_SUMMARY, p=dist[k][\'P\'] / dist[k][\'P\'].sum()) for k in keys}\n    strata = {\'B_summary\': keys, \'B_same_manufacturer\': [k for k in keys if k.startswith(\'set_01\')], \'B_cross_manufacturer\': [k for k in keys if not k.startswith(\'set_01\')]}\n    unif = np.full(MC_SUMMARY, 1.0 / MC_SUMMARY)\n    for h, (a, b) in CONTRASTS.items():\n        for sn, ks in strata.items():\n            d = np.mean([dist[k][\'ap\'][a][draws[k]] - dist[k][\'ap\'][b][draws[k]] for k in ks], 0)\n            R[\'summary\'].setdefault(h, {})[sn] = {\'point_full_sample\': round(float(np.mean([R[\'cells\'][k][\'point\'][h] for k in ks])), 4), **summarise(d, unif)}\n    R[\'holm_B_summary\'] = holm({h: max(R[\'summary\'][h][\'B_summary\'][\'p_exact_two_sided\'], 1.0 / MC_SUMMARY) for h in CONTRASTS})\n    json.dump(R, open(os.path.join(OUT, \'gat_stats.json\'), \'w\'), indent=1)\n    L = [\'cell\\t\' + \'\\t\'.join(MODELS)]\n    for k, c in R[\'cells\'].items():\n        L.append(k + \'\\t\' + \'\\t\'.join(f"{c[\'models\'][m][\'ap_mean\']:.4f}±{c[\'models\'][m][\'ap_sd\']:.4f}" for m in MODELS))\n    L.append(\'\\ncontrast\\tstratum\\tpoint\\tci95\\tci90\\tdecision\\tp_exact\\tholm(B summary)\')\n    for h, dd in R[\'summary\'].items():\n        for sn, v in dd.items():\n            L.append(f"{h}\\t{sn}\\t{v[\'point_full_sample\']:+.4f}\\t[{v[\'ci95\'][0]:+.4f},{v[\'ci95\'][1]:+.4f}]\\t[{v[\'ci90\'][0]:+.4f},{v[\'ci90\'][1]:+.4f}]\\t{v[\'decision\']}\\t{v[\'p_exact_two_sided\']:.4f}\\t"\n                     + (f"{R[\'holm_B_summary\'][h]:.4f}" if sn == \'B_summary\' else \'\'))\n    L.append(\'\\nper-cell exact bootstrap\')\n    for k, c in R[\'cells\'].items():\n        for h, v in c[\'exact_bootstrap\'].items():\n            L.append(f"{k}\\t{h}\\t{v[\'point_full_sample\']:+.4f}\\t[{v[\'ci95\'][0]:+.4f},{v[\'ci95\'][1]:+.4f}]\\t{v[\'decision\']}\\tp {v[\'p_exact_two_sided\']:.4f}")\n    L.append(\'\\noperating points (GAT, GAT_rewired): macroF1 | recall@1 | FA/h@1 | recall@5 | FA/h@5\')\n    for k, c in R[\'cells\'].items():\n        for m in (\'GAT\', \'GAT_rewired\'):\n            v = c[\'models\'][m]\n            L.append(f"{k}\\t{m}\\t{v[\'macro_f1_mean\']:.3f}\\t{v[\'recall_at_1FAh_mean\']:.3f}\\t{v[\'achieved_FAh_at_1FAh_mean\']:.1f}\\t{v[\'recall_at_5FAh_mean\']:.3f}\\t{v[\'achieved_FAh_at_5FAh_mean\']:.1f}")\n    L.append(\'\\nper-family AP (token recordings only): \' + \' | \'.join(MODELS))\n    for k, c in R[\'cells\'].items():\n        for t in c[\'models\'][\'GAT\'][\'per_token\']:\n            L.append(f"{k}\\t{t}\\t{c[\'models\'][\'GAT\'][\'per_token\'][t][\'pos\']}\\t" + \'\\t\'.join(f"{c[\'models\'][m][\'per_token\'][t][\'ap_token_recordings_mean\']:.3f}" for m in MODELS))\n    L.append(\'\\nleave-one-recording-out E1 (GAT - DeepSets)\')\n    for k, c in R[\'cells\'].items():\n        L.append(k + \'\\t\' + \'  \'.join(f"{fn}:{v[\'E1_GAT_minus_DeepSets\']:+.3f}" for fn, v in c[\'leave_one_out\'].items()))\n    open(os.path.join(OUT, \'gat_stats.tsv\'), \'w\').write(\'\\n\'.join(L) + \'\\n\'); print(\'\\n\'.join(L))\n\nif __name__ == \'__main__\':\n    ap = argparse.ArgumentParser(); ap.add_argument(\'--gat\', required=True); ap.add_argument(\'--k05_eval\', required=True); ap.add_argument(\'--out\', default=\'/kaggle/working/gat_stats\')\n    a = ap.parse_args(); main(a.gat, a.k05_eval, a.out)\n', 'exp_audit.py': '"""Stage 5c recording-level audit of the FROZEN k05 outputs. No training, no re-scoring, no exclusion of data.\nPurpose: test whether B-cell conclusions are robust to recording-level imbalance (2 recordings per attack token; one can dominate positives).\nUsage: python exp_audit.py --eval <k05 eval dir> --out /kaggle/working/audit"""\nimport os, sys, json, glob, argparse, itertools, math\nimport numpy as np\nfrom sklearn.metrics import average_precision_score\n\nMODELS = [\'Rule\', \'LightGBM\', \'LightGBM_S\', \'DeepSets\', \'GRU\', \'GraphSAGE\', \'GraphSAGE_rewired\']\nCONTRASTS = {\'H1_GS_minus_DeepSets\': (\'GraphSAGE\', \'DeepSets\'), \'H1a_GS_minus_rewired\': (\'GraphSAGE\', \'GraphSAGE_rewired\'),\n             \'H2_GS_minus_GRU\': (\'GraphSAGE\', \'GRU\'), \'H3_GS_minus_LGBS\': (\'GraphSAGE\', \'LightGBM_S\'),\n             \'ladder_LGBS_minus_LGB\': (\'LightGBM_S\', \'LightGBM\')}\nMARGIN = 0.02; MC_SUMMARY = 200000; MC_SEED = 20260917\n\ndef ap_prep(s, y, fid, nfiles):\n    o = np.argsort(-s, kind=\'stable\'); ss = s[o]; ys = y[o].astype(np.float64); fs = fid[o]\n    ends = np.r_[np.flatnonzero(ss[1:] != ss[:-1]), len(ss) - 1]\n    CP = np.zeros((nfiles, len(ends))); CN = np.zeros((nfiles, len(ends)))\n    for f in range(nfiles):\n        m = fs == f\n        CP[f] = np.cumsum(m * ys)[ends]; CN[f] = np.cumsum(m * (1 - ys))[ends]\n    return CP, CN\n\ndef ap_w(CP, CN, W, chunk=64):\n    out = np.empty(len(W))\n    for c in range(0, len(W), chunk):\n        w = W[c:c + chunk]\n        tp = w @ CP; fp = w @ CN; P = tp[:, -1:]; den = tp + fp\n        prec = np.where(den > 0, tp / np.where(den > 0, den, 1), 0.0)\n        rec = np.where(P > 0, tp / np.where(P > 0, P, 1), 0.0)\n        ap = (np.diff(rec, axis=1, prepend=0.0) * prec).sum(1)\n        out[c:c + chunk] = np.where(P[:, 0] > 0, ap, np.nan)\n    return out\n\ndef exact_token_resamples(tokens):\n    """Every distinct token-stratified bootstrap outcome with its probability. For a token with k recordings, the\n    multiset of k draws with replacement; probability = multinomial weight / k**k."""\n    tokens = np.asarray(tokens); per = []\n    for t in sorted(set(tokens.tolist())):\n        idx = np.flatnonzero(tokens == t); k = len(idx); opts = {}\n        for draw in itertools.product(range(k), repeat=k):\n            cnt = tuple(sorted(np.bincount(draw, minlength=k).tolist(), reverse=False))\n            key = tuple(np.bincount(draw, minlength=k).tolist())\n            opts[key] = opts.get(key, 0) + 1\n        per.append([(idx, np.array(key, float), c / float(k ** k)) for key, c in opts.items()])\n    W = []; P = []\n    for combo in itertools.product(*per):\n        w = np.zeros(len(tokens)); p = 1.0\n        for idx, cnt, pr in combo:\n            w[idx] = cnt; p *= pr\n        W.append(w); P.append(p)\n    return np.array(W), np.array(P)\n\ndef wq(vals, probs, q):\n    o = np.argsort(vals); v = vals[o]; c = np.cumsum(probs[o]); c /= c[-1]\n    return float(v[np.searchsorted(c, q, side=\'left\').clip(0, len(v) - 1)])\n\ndef summarise(vals, probs):\n    m = np.isfinite(vals); vals, probs = vals[m], probs[m] / probs[m].sum()\n    lo95, hi95, lo90, hi90 = wq(vals, probs, .025), wq(vals, probs, .975), wq(vals, probs, .05), wq(vals, probs, .95)\n    if lo95 > 0: d = \'superior\'\n    elif hi95 < 0: d = \'inferior\'\n    elif lo90 >= -MARGIN and hi90 <= MARGIN: d = \'practically_equivalent\'\n    else: d = \'inconclusive\'\n    p = min(1.0, 2 * min(probs[vals <= 0].sum(), probs[vals >= 0].sum()))\n    return {\'mean_over_resamples\': float((vals * probs).sum()), \'ci95\': [lo95, hi95], \'ci90\': [lo90, hi90],\n            \'decision\': d, \'p_exact_two_sided\': float(p), \'n_distinct_resamples\': int(len(vals))}\n\ndef main(EVAL, OUT):\n    os.makedirs(OUT, exist_ok=True); R = {\'cells\': {}, \'summary\': {}, \'method\': {\n        \'per_recording_ap\': \'AP computed on that recording alone (diagnostic only; unstable at low positive counts)\',\n        \'leave_one_out\': \'cell AP recomputed with that recording removed (replaces the ill-defined per-recording contribution to a global ranking metric)\',\n        \'exact_bootstrap\': \'complete enumeration of the token-stratified recording bootstrap with exact probabilities (replaces 2,000 random draws; same estimator)\',\n        \'no_exclusions\': \'no recording is dropped from the frozen results; leave-one-out is diagnostic\'}}\n    per_cell_dist = {}\n    for path in sorted(glob.glob(os.path.join(EVAL, \'set_*\', \'test_02_scores.npz\'))):\n        st = path.split(os.sep)[-2]; key = f\'{st}/test_02\'\n        Z = np.load(path); meta = json.load(open(path.replace(\'_scores.npz\', \'_meta.json\')))\n        y = Z[\'y\'].astype(np.int8); fid = Z[\'file_id\'].astype(np.int64); files = meta[\'files\']; nf = len(files)\n        tokens = [f[\'token\'] for f in files]\n        S = {m: Z[f\'score_{m}\'].astype(np.float64) for m in MODELS}\n        cell = {\'files\': [], \'aggregate\': {}, \'leave_one_out\': {}, \'exact_bootstrap\': {}, \'seed_vs_recording_variability\': {}}\n        tot_pos = int(y.sum())\n        for i, f in enumerate(files):\n            ix = fid == i; rec = {\'file\': f[\'file\'], \'token\': f[\'token\'], \'windows\': int(ix.sum()), \'pos\': int(y[ix].sum()),\n                                  \'share_of_cell_positives\': round(float(y[ix].sum()) / max(tot_pos, 1), 4),\n                                  \'prevalence\': round(float(y[ix].mean()), 5), \'hours\': round(f[\'hours\'], 4), \'ap_mean\': {}, \'ap_sd\': {}}\n            for m in MODELS:\n                a = [average_precision_score(y[ix], s[ix]) if y[ix].sum() > 0 else float(\'nan\') for s in S[m]]\n                rec[\'ap_mean\'][m] = round(float(np.mean(a)), 4); rec[\'ap_sd\'][m] = round(float(np.std(a, ddof=1)) if len(a) > 1 else 0.0, 4)\n            rec[\'contrasts\'] = {h: round(rec[\'ap_mean\'][a] - rec[\'ap_mean\'][b], 4) for h, (a, b) in CONTRASTS.items()}\n            cell[\'files\'].append(rec)\n        agg = {m: float(np.mean([average_precision_score(y, s) for s in S[m]])) for m in MODELS}\n        cell[\'aggregate\'] = {\'ap_mean\': {m: round(agg[m], 4) for m in MODELS},\n                             \'contrasts\': {h: round(agg[a] - agg[b], 4) for h, (a, b) in CONTRASTS.items()},\n                             \'windows\': int(len(y)), \'pos\': tot_pos, \'prevalence\': round(float(y.mean()), 5)}\n        for i, f in enumerate(files):   # leave-one-recording-out\n            keep = fid != i\n            if y[keep].sum() == 0: continue\n            a2 = {m: float(np.mean([average_precision_score(y[keep], s[keep]) for s in S[m]])) for m in MODELS}\n            cell[\'leave_one_out\'][f[\'file\']] = {\'ap_mean\': {m: round(a2[m], 4) for m in MODELS},\n                                                \'contrasts\': {h: round(a2[a] - a2[b], 4) for h, (a, b) in CONTRASTS.items()},\n                                                \'delta_vs_full\': {h: round((a2[a] - a2[b]) - (agg[a] - agg[b]), 4) for h, (a, b) in CONTRASTS.items()}}\n        W, P = exact_token_resamples(tokens)\n        B = {}\n        for m in MODELS:\n            acc = np.zeros(len(W))\n            for s in S[m]:\n                CP, CN = ap_prep(s, y, fid, nf)\n                chk = ap_w(CP, CN, np.ones((1, nf)))[0]\n                assert abs(chk - average_precision_score(y, s)) < 1e-9\n                acc += ap_w(CP, CN, W)\n            B[m] = acc / len(S[m])\n        per_cell_dist[key] = {\'W_prob\': P, \'ap\': B}\n        for h, (a, b) in CONTRASTS.items():\n            cell[\'exact_bootstrap\'][h] = {\'point_full_sample\': round(agg[a] - agg[b], 4), **summarise(B[a] - B[b], P)}\n        for m in MODELS:\n            sd_seed = float(np.std([average_precision_score(y, s) for s in S[m]], ddof=1)) if len(S[m]) > 1 else 0.0\n            sd_rec = float(np.sqrt(((B[m] - (B[m] * P).sum()) ** 2 * P).sum()))\n            cell[\'seed_vs_recording_variability\'][m] = {\'sd_across_seeds\': round(sd_seed, 4), \'sd_across_recording_resamples\': round(sd_rec, 4)}\n        R[\'cells\'][key] = cell\n        print(key, \'exact resamples\', len(W), \'aggregate\', cell[\'aggregate\'][\'contrasts\'], flush=True)\n    keys = sorted(per_cell_dist)\n    rng = np.random.default_rng(MC_SEED)\n    draws = {k: rng.choice(len(per_cell_dist[k][\'W_prob\']), size=MC_SUMMARY, p=per_cell_dist[k][\'W_prob\'] / per_cell_dist[k][\'W_prob\'].sum()) for k in keys}\n    strata = {\'B_summary\': keys, \'B_same_manufacturer\': [k for k in keys if k.startswith(\'set_01\')], \'B_cross_manufacturer\': [k for k in keys if not k.startswith(\'set_01\')]}\n    unif = np.full(MC_SUMMARY, 1.0 / MC_SUMMARY)\n    for h, (a, b) in CONTRASTS.items():\n        for sname, ks in strata.items():\n            d = np.mean([per_cell_dist[k][\'ap\'][a][draws[k]] - per_cell_dist[k][\'ap\'][b][draws[k]] for k in ks], 0)\n            pt = float(np.mean([R[\'cells\'][k][\'aggregate\'][\'contrasts\'][h] for k in ks]))\n            R[\'summary\'].setdefault(h, {})[sname] = {\'point_full_sample\': round(pt, 4), \'cells\': ks,\n                                                     **summarise(d, unif), \'note\': f\'{MC_SUMMARY} draws from the EXACT per-cell resample distributions\'}\n    json.dump(R, open(os.path.join(OUT, \'audit.json\'), \'w\'), indent=1)\n    # readable tables\n    L = []\n    for k, c in R[\'cells\'].items():\n        L.append(f"\\n== {k}  windows {c[\'aggregate\'][\'windows\']}  pos {c[\'aggregate\'][\'pos\']}  prevalence {c[\'aggregate\'][\'prevalence\']}")\n        L.append(\'recording\\ttoken\\twin\\tpos\\tposshare\\t\' + \'\\t\'.join(MODELS) + \'\\tGS-DS\\tGS-rew\')\n        for f in c[\'files\']:\n            L.append(f"{f[\'file\']}\\t{f[\'token\']}\\t{f[\'windows\']}\\t{f[\'pos\']}\\t{f[\'share_of_cell_positives\']}\\t" + \'\\t\'.join(f"{f[\'ap_mean\'][m]:.3f}" for m in MODELS)\n                     + f"\\t{f[\'contrasts\'][\'H1_GS_minus_DeepSets\']:+.3f}\\t{f[\'contrasts\'][\'H1a_GS_minus_rewired\']:+.3f}")\n        L.append(\'AGGREGATE\\t\\t\\t\\t\\t\' + \'\\t\'.join(f"{c[\'aggregate\'][\'ap_mean\'][m]:.3f}" for m in MODELS)\n                 + f"\\t{c[\'aggregate\'][\'contrasts\'][\'H1_GS_minus_DeepSets\']:+.3f}\\t{c[\'aggregate\'][\'contrasts\'][\'H1a_GS_minus_rewired\']:+.3f}")\n        L.append(\'leave-one-out (cell AP without that recording) — GS-DS / GS-rewired, and change vs full cell:\')\n        for fn, v in c[\'leave_one_out\'].items():\n            L.append(f"  drop {fn}\\tGS-DS {v[\'contrasts\'][\'H1_GS_minus_DeepSets\']:+.3f} ({v[\'delta_vs_full\'][\'H1_GS_minus_DeepSets\']:+.3f})"\n                     f"\\tGS-rew {v[\'contrasts\'][\'H1a_GS_minus_rewired\']:+.3f} ({v[\'delta_vs_full\'][\'H1a_GS_minus_rewired\']:+.3f})"\n                     f"\\tGS-LGB+S {v[\'contrasts\'][\'H3_GS_minus_LGBS\']:+.3f} ({v[\'delta_vs_full\'][\'H3_GS_minus_LGBS\']:+.3f})")\n        L.append(\'exact token-stratified bootstrap:\')\n        for h, v in c[\'exact_bootstrap\'].items():\n            L.append(f"  {h}\\tpoint {v[\'point_full_sample\']:+.4f}\\tmean {v[\'mean_over_resamples\']:+.4f}\\tci95 [{v[\'ci95\'][0]:+.4f},{v[\'ci95\'][1]:+.4f}]\\tci90 [{v[\'ci90\'][0]:+.4f},{v[\'ci90\'][1]:+.4f}]\\t{v[\'decision\']}\\tp {v[\'p_exact_two_sided\']:.4f}\\tN {v[\'n_distinct_resamples\']}")\n        L.append(\'variability (SD across seeds | SD across recording resamples):\')\n        L.append(\'  \' + \'  \'.join(f"{m} {v[\'sd_across_seeds\']:.3f}|{v[\'sd_across_recording_resamples\']:.3f}" for m, v in c[\'seed_vs_recording_variability\'].items()))\n    L.append(\'\\n== B summary / strata (exact per-cell distributions)\')\n    L.append(\'contrast\\tstratum\\tpoint\\tmean\\tci95\\tci90\\tdecision\\tp\')\n    for h, dd in R[\'summary\'].items():\n        for sname, v in dd.items():\n            L.append(f"{h}\\t{sname}\\t{v[\'point_full_sample\']:+.4f}\\t{v[\'mean_over_resamples\']:+.4f}\\t[{v[\'ci95\'][0]:+.4f},{v[\'ci95\'][1]:+.4f}]\\t[{v[\'ci90\'][0]:+.4f},{v[\'ci90\'][1]:+.4f}]\\t{v[\'decision\']}\\t{v[\'p_exact_two_sided\']:.4f}")\n    open(os.path.join(OUT, \'audit.tsv\'), \'w\').write(\'\\n\'.join(L) + \'\\n\'); print(\'\\n\'.join(L))\n\nif __name__ == \'__main__\':\n    ap = argparse.ArgumentParser(); ap.add_argument(\'--eval\', required=True); ap.add_argument(\'--out\', default=\'/kaggle/working/audit\'); a = ap.parse_args()\n    main(a.eval, a.out)\n'}
for n, t in FILES.items():
    open(os.path.join('/kaggle/working/code', n), 'w').write(t)
print(sorted(os.listdir('/kaggle/working/code')))
import glob
print('zip:', glob.glob('/kaggle/input/**/can-train-and-test-v1.zip', recursive=True))
print('hashes:', glob.glob('/kaggle/input/**/file_hashes_sha256.csv', recursive=True))
print('k03 meta:', len(glob.glob('/kaggle/input/**/tune/set_0*/meta.json', recursive=True)))
K05 = sorted(glob.glob('/kaggle/input/**/eval/set_0*/test_02_scores.npz', recursive=True)); print('k05 B-cell scores:', K05)
K05_EVAL = os.path.dirname(os.path.dirname(K05[0])); print('K05_EVAL', K05_EVAL)


In [ ]:
# correctness checks on GPU before the run
import sys, torch; sys.path.insert(0, '/kaggle/working/code')
import models
torch.manual_seed(0); dev = 'cuda:0'; B, N = 6, 64
H = torch.randn(B, N, 13, device=dev)
src = torch.randint(0, 20, (B, 63), device=dev, dtype=torch.uint8); dst = torch.randint(0, 20, (B, 63), device=dev, dtype=torch.uint8)
A = models.build_adj(src, dst, B, dev)
sage = models.SAGELayer(13, 32).to(dev); gat = models.GATLayer(13, 32, 4).to(dev)
gat.self_lin.load_state_dict(sage.self_lin.state_dict()); gat.nei_lin.load_state_dict(sage.nei_lin.state_dict())
with torch.no_grad():
    gat.a_src.zero_(); gat.a_dst.zero_()
    d = (gat(H, A) - sage(H, A)).abs().max().item()
print('GAT with a=0 vs GraphSAGE layer, max |diff| =', d); assert d < 1e-4
m = models.GAT(13, 32, 4).to(dev); b = {'node': H, 'nmask': torch.ones(B, N, dtype=torch.bool, device=dev), 'glob': torch.randn(B, 4, device=dev), 'adj': A}
out = m(b); out.sum().backward(); assert torch.isfinite(out).all(); print('forward/backward ok', out.shape)
for h in (32, 64):
    gs = models.nparams(models.GraphSAGE(13, h, 4)); cand = min(range(4, 513, 4), key=lambda x: abs(models.nparams(models.GAT(13, x, 4)) - gs))
    print('h_sage', h, 'GraphSAGE params', gs, '-> GAT width', cand, 'params', models.nparams(models.GAT(13, cand, 4)))


In [ ]:
# pipeline smoke test (2 configs, 1 epoch, 1 seed, 2 test files; NO test metric computed)
import subprocess, sys
r = subprocess.run([sys.executable, '/kaggle/working/code/exp_gat.py', '--sets', 'set_01', '--device', 'cuda:0', '--smoke'], capture_output=True, text=True)
print(r.stdout[-3000:]); print(r.stderr[-3000:])
log = open('/kaggle/working/gat_smoke/log_set_01.txt').read(); assert 'FATAL' not in log and 'DONE' in log, 'smoke failed'
import shutil; shutil.rmtree('/kaggle/working/gat_smoke'); print('smoke OK')


In [ ]:
# main run: two GPUs in parallel
import subprocess, sys, time, glob
t0 = time.time()
pA = subprocess.Popen([sys.executable, '/kaggle/working/code/exp_gat.py', '--sets', 'set_01,set_03', '--device', 'cuda:0'])
pB = subprocess.Popen([sys.executable, '/kaggle/working/code/exp_gat.py', '--sets', 'set_02,set_04', '--device', 'cuda:1'])
print('exit codes', pA.wait(), pB.wait(), 'hours', round((time.time() - t0) / 3600, 2))
print(sorted(glob.glob('/kaggle/working/gat/set_0*/DONE')))
logs = ''.join(open(p).read() for p in glob.glob('/kaggle/working/gat/log_*.txt'))
print('FATAL/ERROR lines:', [l[:300] for l in logs.split('\n') if 'FATAL' in l or 'ERROR' in l])
print('\n'.join(l for l in logs.split('\n') if 'selected' in l or 'identical' in l or 'scored' in l))


In [ ]:
r = subprocess.run([sys.executable, '/kaggle/working/code/gat_stats.py', '--gat', '/kaggle/working/gat', '--k05_eval', K05_EVAL, '--out', '/kaggle/working/gat_stats'], capture_output=True, text=True)
print(r.stdout); print(r.stderr[-3000:]); print('stats exit', r.returncode)
